# HDB Resale Flat Data Engineering Technical Test — Part 1

This notebook is the submission-facing execution walkthrough for the Python ETL pipeline.

It is designed to be placed in the repository under:

```text
hdb_technical_test/
└── notebooks/
    └── hdb_pipeline_submission.ipynb
```

The notebook calls the existing Python modules rather than duplicating the business logic:

```text
src/extract.py
src/profile.py
src/transform.py
src/hashed.py
```

Pipeline sequence:

```text
data.gov.sg
    ↓
extract_processing()
    ↓
data/raw/
    ↓
profile_processing()
    ↓
profiling/ + data/edited/ + data/cleaned/
    ↓
transform_processing()
    ↓
data/transformed/ + data/quarantined/
    ↓
hash_processing()
    ↓
data/hashed/
```

> **Before submission:** run **Kernel → Restart & Run All** (or **Run All**) locally so that the notebook saves the real execution outputs from your repository.


## 1. Environment and project setup

The notebook locates the repository root automatically whether it is opened from the repository root or from the `notebooks/` folder.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()

if (cwd / "src").exists() and (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists() and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the hdb_technical_test project root. "
        "Open this notebook from the repository or notebooks folder."
    )

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
EDITED_DIR = DATA_DIR / "edited"
CLEANED_DIR = DATA_DIR / "cleaned"
TRANSFORMED_DIR = DATA_DIR / "transformed"
QUARANTINED_DIR = DATA_DIR / "quarantined"
HASHED_DIR = DATA_DIR / "hashed"
PROFILING_DIR = PROJECT_ROOT / "profiling"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


## 2. Import pipeline functions

`main.py` orchestrates these same four processing functions. Importing them individually here allows the notebook to show the output of each pipeline stage.


In [ ]:
from extract import extract_processing
from profile import profile_processing
from transform import transform_processing
from hashed import hash_processing

print("Pipeline modules imported successfully.")


## 3. Data extraction

The extraction stage:

1. Calls the data.gov.sg collection metadata API for collection `189`.
2. Retrieves the child dataset IDs.
3. Requests a download URL for each dataset.
4. Downloads each source CSV into `data/raw/`.

The raw source files are preserved as downloaded.


In [ ]:
extract_processing()


### Raw dataset audit

Confirm that the five raw datasets were downloaded and inspect their basic structure.


In [ ]:
raw_files = sorted(RAW_DIR.glob("dataset_*.csv"))

raw_summary = []
for file in raw_files:
    df = pd.read_csv(file)
    raw_summary.append({
        "file": file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns),
    })

raw_summary_df = pd.DataFrame(raw_summary)
raw_summary_df


## 4. Data profiling and schema normalisation

`profile_processing()` generates a `ydata-profiling` HTML report for each source dataset.

Based on profiling, datasets 1 and 5 contain a source `remaining_lease` column. Working copies of these two files are created without that column so that remaining lease can be recalculated consistently later.

The five compatible datasets are then concatenated into:

```text
data/cleaned/master_dataset.csv
```

A profiling report is also generated for the master dataset.


In [ ]:
profile_processing()


### Profiling report outputs


In [ ]:
profile_reports = sorted(PROFILING_DIR.glob("report_*.html"))

pd.DataFrame({
    "profiling_report": [p.name for p in profile_reports],
    "exists": [p.exists() for p in profile_reports],
    "size_kb": [round(p.stat().st_size / 1024, 1) for p in profile_reports],
})


### Master dataset inspection


In [ ]:
master_path = CLEANED_DIR / "master_dataset.csv"
master_df = pd.read_csv(master_path)

print("Master dataset shape:", master_df.shape)
print("\nColumns:")
print(master_df.columns.tolist())

master_df.head()


### Master dataset quality summary

This provides a compact notebook output alongside the detailed HTML profiling report.


In [ ]:
quality_summary = pd.DataFrame({
    "column": master_df.columns,
    "dtype": [str(master_df[c].dtype) for c in master_df.columns],
    "null_count": [int(master_df[c].isna().sum()) for c in master_df.columns],
    "unique_count": [int(master_df[c].nunique(dropna=True)) for c in master_df.columns],
})

quality_summary


## 5. Remaining lease calculation and duplicate handling

`transform_processing()`:

- parses `month` using the `YYYY-MM` format;
- assumes a 99-year HDB lease;
- calculates the number of lease months already used;
- derives remaining years and months;
- creates a human-readable `remaining_lease`;
- sorts records by `resale_price` descending;
- uses all columns except `resale_price` as the composite key;
- retains the higher-price record when duplicate composite keys exist;
- writes lower-priced duplicates to the quarantine output.


In [ ]:
transform_processing()


### Transformed data


In [ ]:
transformed_path = TRANSFORMED_DIR / "transformed_dataset.csv"
transformed_df = pd.read_csv(transformed_path)

print("Transformed dataset shape:", transformed_df.shape)

display_columns = [
    c for c in [
        "month",
        "town",
        "flat_type",
        "block",
        "lease_commence_date",
        "remaining_lease",
        "resale_price",
    ]
    if c in transformed_df.columns
]

transformed_df[display_columns].head(10)


### Quarantined duplicate records


In [ ]:
quarantine_path = QUARANTINED_DIR / "quarantined_dataset.csv"
quarantined_df = pd.read_csv(quarantine_path)

print("Quarantined record count:", len(quarantined_df))
print("Quarantined dataset shape:", quarantined_df.shape)

quarantined_df.head(10)


### Duplicate-resolution verification

After duplicate handling, the transformed dataset should no longer contain duplicate composite keys when `resale_price` is excluded.


In [ ]:
composite_key = [
    column for column in transformed_df.columns
    if column != "resale_price"
]

remaining_duplicates = transformed_df.duplicated(
    subset=composite_key,
    keep=False
).sum()

print("Duplicate composite-key records remaining:", int(remaining_duplicates))


## 6. Resale identifier and SHA-256 hashing

`hash_processing()` builds the resale identifier components and creates an irreversible SHA-256 hash.

The final output is written to:

```text
data/hashed/hashed_dataset.csv
```


In [ ]:
hash_processing()


### Hashed dataset inspection


In [ ]:
hashed_path = HASHED_DIR / "hashed_dataset.csv"
hashed_df = pd.read_csv(hashed_path)

print("Hashed dataset shape:", hashed_df.shape)
print("Hashed identifier null count:", int(hashed_df["hashed_identifier"].isna().sum()))
print("Unique hashed identifiers:", int(hashed_df["hashed_identifier"].nunique()))

hashed_df.head(10)


### SHA-256 format verification

A SHA-256 hexadecimal digest should contain 64 hexadecimal characters.


In [ ]:
hash_length_check = (
    hashed_df["hashed_identifier"]
    .astype(str)
    .str.fullmatch(r"[0-9a-f]{64}")
)

print("Rows with valid SHA-256 hexadecimal format:", int(hash_length_check.sum()))
print("Total rows:", len(hashed_df))
print("All hashes valid:", bool(hash_length_check.all()))


## 7. Final output inventory

This cell provides a concise audit of the mandatory and intermediate files created by the current implementation.


In [ ]:
output_files = {
    "Raw dataset 1": RAW_DIR / "dataset_1.csv",
    "Raw dataset 2": RAW_DIR / "dataset_2.csv",
    "Raw dataset 3": RAW_DIR / "dataset_3.csv",
    "Raw dataset 4": RAW_DIR / "dataset_4.csv",
    "Raw dataset 5": RAW_DIR / "dataset_5.csv",
    "Edited dataset 1": EDITED_DIR / "dataset_edited_1.csv",
    "Edited dataset 5": EDITED_DIR / "dataset_edited_5.csv",
    "Master dataset": CLEANED_DIR / "master_dataset.csv",
    "Transformed dataset": TRANSFORMED_DIR / "transformed_dataset.csv",
    "Quarantined dataset": QUARANTINED_DIR / "quarantined_dataset.csv",
    "Hashed dataset": HASHED_DIR / "hashed_dataset.csv",
    "Master profile": PROFILING_DIR / "report_master.html",
}

inventory = []
for name, path in output_files.items():
    inventory.append({
        "output": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
    })

pd.DataFrame(inventory)


## 8. Pipeline completion summary

The notebook demonstrates the complete execution path of the submitted Python codebase:

```text
Extract
→ Profile / Combine
→ Remaining Lease / Duplicate Handling
→ Resale Identifier / Hash
```

The detailed profiling reports remain available under `profiling/`, while the notebook shows representative dataframe outputs and validation checks for reviewer convenience.

### Submission reminder

Before committing this notebook to GitHub:

1. Activate the project's `.venv`.
2. Install `requirements.txt`.
3. Open this notebook from the repository.
4. Select the `.venv` Python kernel.
5. Run **Restart & Run All**.
6. Confirm all cells complete without errors.
7. Save the notebook so the executed cell outputs are embedded in the `.ipynb` file.
